# Assignment 2: ML Model Training & Evaluation

## Part 0: Data Cleaning

### Step 1: Data Ingestion and Storage

In [ ]:
from urllib.request import urlretrieve
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

files = [
    ("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", raw_dir/"yellow_taxi_data.parquet"),
    ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", raw_dir/"taxi_zone_lookup.csv"),
]

for url, filename in files:
    urlretrieve(url, filename)

print("Done!")

In [ ]:
import polars as pl

df = pl.read_parquet(raw_dir/'yellow_taxi_data.parquet')
df.head()

In [ ]:
import sys
expected_result = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "payment_type"]

try:
    missing = [m for m in expected_result if m not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    print("All columns present")

    if df.schema.get("tpep_pickup_datetime") != pl.Datetime:
        raise TypeError(f"tpep_pickup_datetime is {df.schema.get('tpep_pickup_datetime')}, expected Datetime")
    print("tpep_pickup_datetime datatype is: ", df.schema.get("tpep_pickup_datetime"))

    if df.schema.get("tpep_dropoff_datetime") != pl.Datetime:
        raise TypeError(f"tpep_dropoff_datetime is {df.schema.get('tpep_dropoff_datetime')}, expected Datetime")
    print("tpep_dropoff_datetime datatype is: ", df.schema.get("tpep_dropoff_datetime"))

    print(f'Number of rows: {len(df):,}')
    print(f'Number of columns: {len(df.columns)}')
    print('\nColumn names and types:')
    print(df.schema)

except Exception as e:
    print(f"Data Validation failed: {e}")
    sys.exit(1)


### Step 2: Data Transformation & Analysis

In [ ]:
# ensuring the data is clean to avoid issues in analysis and visualizations
# removing nulls values in all columns
df_clean = df.drop_nulls()
print(f'Number of rows after dropping nulls: {len(df_clean):,}')
print(f'Number of rows removed after dropping nulls: {len(df) - len(df_clean):,}')

#filtering invalid trips: trips with zero or negative distance, negative fares, or fares exceeding $500
df_invalid = (
    df_clean
    .filter((pl.col("fare_amount") > 0) & (pl.col("fare_amount") <= 500))
    .filter(pl.col("trip_distance") > 0)
)
print(f'Number of rows after filtering invalid trips: {len(df_invalid):,}')
print(f'Number of rows removed after filtering invalid trips: {len(df_clean) - len(df_invalid):,}')

#removing trips where dropoff time is before pickup time
df_final = df_invalid.filter(pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"))
print(f'Number of rows after filtering dropoff before pickup: {len(df_final):,}')
print(f'Number of rows removed after filtering dropoff before pickup: {len(df_invalid) - len(df_final):,}')


In [ ]:
#adding new column for trip duration in minutes
enriched = df_final.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60).alias('trip_duration_minutes'),
])

enriched.select(pl.col("trip_duration_minutes")).head()

In [ ]:
#adding new column for trip speed
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
        .otherwise(None)
        .alias('trip_speed_mph')
    )
])

enriched.select(pl.col("trip_speed_mph"))

In [ ]:
#adding new columns for pickup hour and weekday
enriched = enriched.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    pl.col('tpep_pickup_datetime').dt.weekday().alias('pickup_weekday')
])

enriched.select(pl.col("pickup_hour"), pl.col("pickup_weekday")).head()

In [ ]:
print(enriched.schema)

In [ ]:
print(enriched["payment_type"])

In [ ]:
# filtering the dataset to include only credit card transactions for the modeling tasks
enriched = enriched.filter(pl.col("payment_type").is_in([1]))


In [ ]:
print(enriched["payment_type"])

In [ ]:
print(len(enriched))    #dropped from 2.9 mil to 2.2 mil after filtering for credit card transactions

In [ ]:
enriched.write_parquet(raw_dir/'yellow_taxi_data_enriched.parquet')

## Part 1: Data Preprocessing & Feature Engineering

### 1. Feature Engineering

In [ ]:
#temporal features

enriched = enriched.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),
    (pl.col('tpep_pickup_datetime').dt.weekday() - 1).alias('pickup_day_of_week')
])

#days of the week 0 - 6 so weekend is 5 and 6
enriched = enriched.with_columns(
    (pl.col('pickup_day_of_week') >= 5).alias('is_weekend')
)

enriched.select(["tpep_pickup_datetime","pickup_hour","pickup_day_of_week","is_weekend"]).sample(n=5)

In [ ]:
#trip features

#trip duration minutes
enriched = enriched.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 60).alias('trip_duration_minutes')
])

#trip speed mph
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
        .otherwise(None)
        .alias('trip_speed_mph')
    )
])

#log trip distance
enriched = enriched.with_columns([
    pl.col("trip_distance").log1p().alias("log_trip_distance")
])

enriched.sample(n=5)

In [ ]:
# fare features
#fare_per_mile
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_distance") > 0)
        .then (pl.col("fare_amount") / pl.col("trip_distance"))
        .otherwise(None)
        .alias('fare_per_mile')
    )
])

#fare_per_minute
enriched = enriched.with_columns([
    (
        pl.when(pl.col("trip_duration_minutes") > 0)
        .then(pl.col("fare_amount") / pl.col("trip_duration_minutes"))
        .otherwise(None)
        .alias('fare_per_minute')
    )
])

enriched.sample(n=5)

In [ ]:
df = pl.read_csv(raw_dir/'taxi_zone_lookup.csv')
df.head()

In [ ]:
#Zone features

# read zone table
zone_lookup = pl.read_csv(raw_dir/'taxi_zone_lookup.csv')

#join the zone lookup table to enriched table
enriched = enriched.join(
    zone_lookup.select([
        pl.col("LocationID"),
        pl.col("Borough")
    ]),
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
).rename({'Borough': 'pickup_borough'})

enriched = enriched.join(
    zone_lookup.select([
        pl.col("LocationID"),
        pl.col("Borough")
    ]),
    left_on='DOLocationID',
    right_on='LocationID',
    how='left'
).rename({'Borough': 'dropoff_borough'})

enriched.sample(n=6)

In [ ]:
# convert from polars to pandas
import pandas as pd

enriched_pd = enriched.to_pandas()

enriched_pd.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
#ensure a regular dense array is returned and sklearn do not crash with unknown category
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

#look at columns and learn the categories
encoded_2d_array = ohe.fit_transform(enriched_pd[["pickup_borough", "dropoff_borough"]])

# to avoid the new columns being numbered 0,1,2...
encoded_cols = ohe.get_feature_names_out(["pickup_borough", "dropoff_borough"])

#turn encoded output to pandas table and add it to the enriched_pd dataframe
encoded_df = pd.DataFrame(encoded_2d_array, columns=encoded_cols, index=enriched_pd.index)
enriched_pd = pd.concat([enriched_pd, encoded_df], axis=1)

enriched_pd.sample(n=5)

### 2. Target Variable Creation

In [ ]:
# tip_amount (continuous, for regression)
y_reg = enriched_pd["tip_amount"]

#high_tip (binary: 1 if tip_amount > 20% of fare_amount, 0 otherwise, for classification
enriched_pd["high_tip"] = (
    (enriched_pd["tip_amount"] > 0) &
    (enriched_pd["tip_amount"] > 0.2 * enriched_pd["fare_amount"])
).astype(int)

y_clf = enriched_pd["high_tip"]

enriched_pd.sample(n=5)

In [ ]:

enriched.select(["tip_amount", "fare_amount"]).sample(n=5)
print(enriched_pd["tip_amount"].max())
print(enriched_pd["fare_amount"].max())

### 3. Data Splitting & Scaling

In [ ]:
#check max speed
print(f'Max speed: {enriched_pd["trip_speed_mph"].max()}')

In [ ]:
# Removing outliers
enriched_pd = enriched_pd[(enriched_pd["trip_speed_mph"] <= 100) & (enriched_pd["trip_speed_mph"] > 1)]
print(f'Max speed: {enriched_pd["trip_speed_mph"].max()}')
print(f'Min speed: {enriched_pd["trip_speed_mph"].min()}')

In [ ]:
#Dropping columns
columns_to_drop = [
    "RatecodeID",
    "trip_distance",
    "total_amount"
    ]

enriched_pd = enriched_pd.drop(columns_to_drop, axis=1)

In [ ]:
# Split data into training (70%), validation (15%), and test (15%) sets using stratified sampling for the classification target
from sklearn.model_selection import train_test_split

X= enriched_pd.drop(["tip_amount", "high_tip"], axis=1)
y_reg = enriched_pd["tip_amount"]
y_clf = enriched_pd["high_tip"]

X_train, X_temp, y_train_reg, y_temp_reg, y_train, y_temp = train_test_split(
    X, y_reg, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

X_val, X_test, y_val_reg, y_test_reg, y_val, y_test = train_test_split(
    X_temp, y_temp_reg, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [ ]:
#identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()

print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')

In [ ]:
# Apply StandardScaler to numeric features; fit on training data only
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

#numeric pipeline: impute missing values then scale
numeric_transformer = Pipeline(steps=[
  ('imputer', SimpleImputer(strategy='median')),
  ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
transformers=[
  ('num', numeric_transformer, numeric_features),
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)


In [ ]:
included_features = numeric_features
excluded_by_preprocessor = [col for col in X.columns if col not in included_features]

print("Included in preprocessing:")
print(included_features)

print("\nExcluded by preprocessing:")
print(excluded_by_preprocessor)

In [ ]:
#Document the number of samples in each split and the class distribution of high_tip in each split
print("Number of samples in each split:")
print("Training set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

print("\nClass distribution of high_tip in Training set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nClass distribution of high_tip in Validation set:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nClass distribution of high_tip in Test set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

#### Print a summary of feature names, types, and any features excluded from modeling (and why)

In [ ]:
print("Feature names used for modeling:")
print(numeric_features)
print("Number of features used:", len(numeric_features))

In [ ]:
print("\nFeature data types:")
print(X[numeric_features].dtypes)
# print(X.dtypes)

print("\nCount of each data type:")
print(X.dtypes.value_counts())

In [ ]:
print(f"\nNumeric features ({len(numeric_features)}):")
print(numeric_features)

print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)

In [ ]:
excluded_features = [col for col in X.columns if col not in numeric_features]

print("\nExcluded features:")
print(excluded_features)
print("Number of excluded features:", len(excluded_features))

In [ ]:
print("\nExcluded features:")
print("\ntip_amount was excluded because it is a regression target and would cause target leakage")
print("\nhigh_tip was excluded because it is a classification target and would cause target leakage")
print("\nVendorID was excluded it is an identifier and not included in the numeric preprocessing pipepline")
print("\ntpep_pickup_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline")
print("\ntpep_dropoff_datetime was excluded because it is a date and not included in the numeric preprocessing pipepline")
print("\nstore_and_fwd_flag, is_weekend, pickup_borough and dropoff_borough was excluded because it is categorical")
print("\nPULocationID was excluded because it is a identifier")
print("\nDOLocationID was excluded because it is a identifier")
print("\npickup_hour, pickup_weekday and pickup_day_of_week was excluded because the data type is int8 and only data type int64 and float64 was included in the pipeline")

## Part 2: Model Training & Tuning

### Baseline Models

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import numpy as np
import pandas as pd

In [ ]:
#define regression models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
   }


In [ ]:
reg_results = {}

for name, reg_model in reg_models.items():
    print(f"Training {name}...")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", reg_model)
    ])

    # pipe.fit(X_train_sample, y_train_reg_sample)
    pipe.fit(X_train, y_train_reg)
    y_val_pred = pipe.predict(X_val)

    reg_results[name] = {
        "MAE": mean_absolute_error(y_val_reg, y_val_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val_reg, y_val_pred)),
        "R2": r2_score(y_val_reg, y_val_pred)
    }
print("Regression model training completed.")

In [ ]:
print("Regression Model Performance:")
for name, metrics in reg_results.items():
    print(f"{name}:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value}")
    print()

In [ ]:
# define classification models
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, random_state=42)
}

In [ ]:
clf_results = {}

for name, clf_model in clf_models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf_model)
    ])

    # pipe.fit(X_train_sample, y_train_sample)
    pipe.fit(X_train, y_train)
    y_val_pred = pipe.predict(X_val)
    y_val_prob = pipe.predict_proba(X_val)[:, 1]

    clf_results[name] = {
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred),
        "AUC-ROC": roc_auc_score(y_val, y_val_prob)
    }

In [ ]:
print("Classification Model Performance:")
for name, metrics in clf_results.items():
    print(f"{name}:")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value}")
    print()

### 5. Hyperparameter Tuning

In [ ]:
#Using RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
from scipy.stats import randint

### Best Regression Model: Logistic Regression

In [ ]:
# Logistic Regression pipeline
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Hyperparameter search space
param_distributions = {
    "classifier__C": loguniform(1e-4, 1e3),   # regularization strength
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs", "saga"],
    "classifier__class_weight": [None, "balanced"]
}

# Stratified sampling for tuning subset using 500k samples
X_train_tune, _, y_train_tune, _ = train_test_split(
    X_train, y_train, train_size=500000, random_state=42, stratify=y_train
)

# Randomized search
logreg_search = RandomizedSearchCV(
    estimator=logreg_pipeline, param_distributions=param_distributions, n_iter=50, cv=5, scoring="f1", n_jobs=-1, random_state=42, verbose=2
)

logreg_search.fit(X_train_tune, y_train_tune)

print("Best Parameters:", logreg_search.best_params_)
print("Best CV Score:", logreg_search.best_score_)

In [ ]:
# Best tuned Logistic Regression model
best_clf_model = logreg_search.best_estimator_

# Predictions on validation set
y_val_pred_tuned = best_clf_model.predict(X_val)
y_val_prob_tuned = best_clf_model.predict_proba(X_val)[:, 1]

# Tuned results
tuned_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred_tuned),
    "Precision": precision_score(y_val, y_val_pred_tuned),
    "Recall": recall_score(y_val, y_val_pred_tuned),
    "F1": f1_score(y_val, y_val_pred_tuned),
    "AUC-ROC": roc_auc_score(y_val, y_val_prob_tuned)
}

# Baseline Logistic Regression results
baseline_metrics = clf_results["Logistic Regression"]

# Create comparison table
table_data = []
for metric in baseline_metrics.keys():
    table_data.append([
        metric,
        f"{baseline_metrics[metric]:.4f}",
        f"{tuned_results[metric]:.4f}"
    ])

# Print formatted table
print("\n" + "="*60)
print(f"{'Metric':<15} {'Baseline':<20} {'Tuned':<20}")
print("="*60)

for row in table_data:
    print(f"{row[0]:<15} {row[1]:<20} {row[2]:<20}")

print("="*60)

### Neural Network

In [ ]:
#classification task
import torch
import torch.nn as nn

# Check if CUDA available, fall back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

#### Designing a feedforward neural network with at least 2 hidden layers for classification task

In [ ]:
#Convert to PyTorch tensors
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
y_train_tensor = torch.FloatTensor(y_train.values).to(device)

X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
y_val_tensor = torch.FloatTensor(y_val.values).to(device)

X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test.values).to(device)

y_train_tensor = torch.FloatTensor(y_train.to_numpy(copy=True)).to(device)
y_val_tensor = torch.FloatTensor(y_val.to_numpy(copy=True)).to(device)
y_test_tensor = torch.FloatTensor(y_test.to_numpy(copy=True)).to(device)

print(f'Training set: {X_train_tensor.shape}')
print(f'Validation set: {X_val_tensor.shape}')
print(f'Test set: {X_test_tensor.shape}')
print(f'Input features: {X_train_tensor.shape[1]}')

#### Define the network architecture


In [ ]:
import torch.nn as nn

class HighTipClassifier(nn.Module):
  def __init__(self, input_size, hidden_sizes=[128, 64], dropout_rate=0.1):
    super(HighTipClassifier, self).__init__()

    #building layers dynamically
    layers = []
    prev_size = input_size

    for hidden_size in hidden_sizes:
      layers.append(nn.Linear(prev_size, hidden_size))
      layers.append(nn.BatchNorm1d(hidden_size))
      layers.append(nn.ReLU())                              # activation function that add non linearity and change negative values to zero
      layers.append(nn.Dropout(dropout_rate))
      prev_size = hidden_size

    layers.append(nn.Linear(prev_size, 1))

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x).squeeze()

#### Instantiate the model and inspect it

In [ ]:
input_size = X_train_tensor.shape[1]
model = HighTipClassifier(input_size, hidden_sizes=[128, 64]).to(device)

print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

#### Implementing proper training with: batch processing using DataLoader, BCEWithLogitsLoss function, Adam Optimizer


#### Batch Processing Using DataLoader

In [ ]:
# Creating a custom Dataset class
from torch.utils.data import Dataset, DataLoader

class HighTipDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X) if not isinstance(X, torch.Tensor) else X
        self.y = torch.FloatTensor(y) if not isinstance(y, torch.Tensor) else y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# Create datasets
train_dataset = HighTipDataset(X_train_scaled, y_train.to_numpy(copy=True))
val_dataset = HighTipDataset(X_val_scaled, y_val.to_numpy(copy=True))

print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

X_batch, y_batch = next(iter(train_loader))
print(f"Batch X shape: {X_batch.shape}")
print(f"Batch y shape: {y_batch.shape}")

#### Defining loss function and optimizer for classification task

In [ ]:
# Binary Cross-Entropy with Logits Loss for binary classification
criterion = nn.BCEWithLogitsLoss()

# Adam optimizer with learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
print(f'Loss function: {criterion}')
print(f'Optimizer: {optimizer}')

#### Update Training to Use DataLoaders

In [ ]:
# Training function with DataLoader
def train_with_loader(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    n_batches = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches

In [ ]:
# Evaluation function with DataLoader
def evaluate_with_loader(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            total_loss += criterion(y_pred, y_batch).item()
            predictions = (torch.sigmoid(y_pred) >= 0.5).float() # applying sigmoid to get probabilities and then thresholding at 0.5 for binary classification
            correct += (predictions == y_batch).sum().item()
            total += len(y_batch)

    return total_loss / len(loader), correct / total

In [ ]:
import copy

# Early stopping configuration
patience = 10
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

# Training loop with early stopping
num_epochs = 40
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss = train_with_loader(model, train_loader, criterion, optimizer, device)

    train_loss_eval, train_acc = evaluate_with_loader(model, train_loader, criterion, device)
    val_loss, val_acc = evaluate_with_loader(model, val_loader, criterion, device)

    train_losses.append(train_loss_eval)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss_eval:.4f}, "
          f"Val Loss={val_loss:.4f}, "
          f"Train Acc={train_acc:.4f}, "
          f"Val Acc={val_acc:.4f}")

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Best model restored with validation loss: {best_val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))


# Loss curves
ax1.plot(train_losses, label='Train Loss', color='blue')
ax1.plot(val_losses, label='Validation Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)


# Accuracy curves
ax2.plot(train_accs, label='Train Accuracy', color='blue')
ax2.plot(val_accs, label='Validation Accuracy', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


#### Reporting the same evaluation (accuracy, precision, recall, F1-score, and AUC-ROC) metrics as the Scikit-learn models for comparison

In [ ]:
#Evaluate final model on validation set and calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch

# Get predictions
model.eval()

all_probs = []  #auc-roc needs probabilities, not binary predictions
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float().cpu()

        all_probs.extend(probs.numpy())
        all_preds.extend(preds.numpy())
        all_labels.extend(y_batch.numpy())

nn_results = {
    "Accuracy": accuracy_score(all_labels, all_preds),
    "Precision": precision_score(all_labels, all_preds),
    "Recall": recall_score(all_labels, all_preds),
    "F1": f1_score(all_labels, all_preds),
    "AUC-ROC": roc_auc_score(all_labels, all_probs)
}

In [ ]:
print("\nComparison with Scikit-learn Models")
print("=" * 100)

# Create comparison table
table_data = []
metrics = list(clf_results["Logistic Regression"].keys())
for metric in metrics:
    table_data.append([
        metric,
        f"{clf_results['Logistic Regression'][metric]:.4f}",
        f"{clf_results['Random Forest Classifier'][metric]:.4f}",
        f"{nn_results[metric]:.4f}"
    ])

print(f"{'Metric':<15} {'Logistic Regression':<25} {'Random Forest':<25} {'Neural Network':<25}")
print("=" * 100)
for row in table_data:
    print(f"{row[0]:<15} {row[1]:<25} {row[2]:<25} {row[3]:<25}")
print("=" * 100)

## Part 3: Model Evaluation & Interpretation

### Comprehensive Evaluation

In [ ]:
# Create a summary table comparing all models across all metrics
reg_summary_df = pd.DataFrame(reg_results).T
print("Regression Model Summary:\n")
print(reg_summary_df)

clf_summary_df = pd.DataFrame(clf_results).T
clf_summary_df.loc["Neural Network"] = nn_results

print("\n\nClassification Model Summary:\n")
print(clf_summary_df)

In [ ]:
#plotting ROC curves for all models on the same figure,
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))

# Logistic Regression ROC
log_reg_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipe.fit(X_train, y_train)
y_test_prob_log = log_reg_pipe.predict_proba(X_test)[:, 1]
fpr_log, tpr_log, _ = roc_curve(y_test, y_test_prob_log)
auc_log = auc(fpr_log, tpr_log)
plt.plot(fpr_log, tpr_log, label=f"Logistic Regression (AUC = {auc_log:.4f})")

# Random Forest Classifier ROC
rf_clf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])
rf_clf_pipe.fit(X_train, y_train)
y_test_prob_rf = rf_clf_pipe.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_test_prob_rf)
auc_rf = auc(fpr_rf, tpr_rf)
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest Classifier (AUC = {auc_rf:.4f})")


# Neural Network ROC
test_dataset = HighTipDataset(X_test_scaled, y_test.to_numpy(copy=True))
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

fpr_nn, tpr_nn, _ = roc_curve(all_labels, all_probs)
auc_nn = auc(fpr_nn, tpr_nn)
plt.plot(fpr_nn, tpr_nn, label=f"Neural Network (AUC = {auc_nn:.4f})")

# Reference line
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Classification Models")
plt.legend()
plt.grid(True)
plt.show()


ROC Curves for Classification Models
The diagram show the Neural Network has the highest AUC score (0.6152), indicating it performed slightly better than the Logistic Regression (0.6057) in the ability to distinguish trips with high-tips from normal tips. However, all models have a relatively low AUC values which suggest it is difficult to predict high tips.

In [ ]:
# Best regression model
best_regressor = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

best_regressor.fit(X_train, y_train_reg)
y_test_pred_reg = best_regressor.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, y_test_pred_reg, alpha=0.3)
plt.xlabel("Actual Tip Amount")
plt.ylabel("Predicted Tip Amount")
plt.title("Predicted vs Actual Tip Amounts - Best Regression Model")

# ideal line
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         linestyle="--")

plt.tight_layout()
plt.show()

Predicted vs Actual Tip Amounts - Best Regression Model
This plot shows that the model performs reasonably well for typical tip amounts due to the high clustering around the diagonal line for smaller values. However, for larger tips the model tends to underestimate the true values, therefore, showing difficulty in predicting high tips. This behaviour is expected due to other factors outside of dataset such as passenger income and quality of service from driver.

In [ ]:
import seaborn as sns

# Residuals
residuals = y_test_reg - y_test_pred_reg

# 1. Residual distribution
plt.figure(figsize=(8, 6))
sns.histplot(residuals, bins=50, kde=True)
plt.xlabel("Residuals")
plt.title("Residual Distribution - Best Regression Model")
plt.show()

# 2. Residuals vs predicted values
plt.figure(figsize=(8, 6))
plt.scatter(y_test_pred_reg, residuals, alpha=0.3)
plt.axhline(0, linestyle="--", color="red")
plt.xlabel("Predicted Tip Amount")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values - Best Regression Model")
plt.show()

Residual Distribution
The tall spike at 0 implies that the model predicts many trips fairly accurately and it being centered at zero suggests that there is no strong bias. While most residuals are small, there are some large outliers this is due to very large tips being hard to predict. However, the model can predict tip amounts reasonably well for most trips.

Residuals vs Predicted Values - Best Regression Model
This shows that the model can predict small tips reasonably well (due to most residuals being clustered around 0 for smaller predicted tip) but it is less accurate when it comes to larger tips (this can be seen in the increase of residuals spread when the predicted tip amount goes up).


### Feature Importance

In [ ]:
#Extract and plot feature importances from the Random Forest model (bar chart, sorted)

# Train Random Forest pipeline
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipe.fit(X_train, y_train)

# Get feature names from preprocessor
feature_names = rf_pipe.named_steps["preprocessor"].get_feature_names_out()

# Get feature importances for Tree-based models
importances = rf_pipe.named_steps["classifier"].feature_importances_

#  Sort and plot top 10 features
feat_imp = pd.Series(importances, index=feature_names)
top_features = feat_imp.nlargest(10)

plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind="barh")
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Extract and interpret coefficients from the Logistic Regression model
# Fit Logistic Regression pipeline
log_reg_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipe.fit(X_train, y_train)

# Get transformed feature names
feature_names = log_reg_pipe.named_steps["preprocessor"].get_feature_names_out()

# Get coefficients
coefficients = log_reg_pipe.named_steps["classifier"].coef_[0]

# Create dataframe
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Sort by absolute magnitude
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values(by="Abs_Coefficient", ascending=False)

print("Top Logistic Regression Coefficients:")
print(coef_df[["Feature", "Coefficient"]].head(15))

### Written Analysis

(a) Which model performed best for each task and why you think this is the case

Regression Model to predict tip_amount: The Linear Regression performed the best. This is supported by it having the lowest MAE (1.2435) and RMSE (2.3812) in comparison to Random Forest Regression. This was influenced by various decisions I made. Firstly, I would like to note initially Random Forest Regression was the better model but after doing additional clean up outside of the requirements Linear Regression performed better. Initially, the max trip speed was revealed to be 41760.0mph so I cut it down to be 100mph or less. Additionally, 'RateCodeID', 'trip_distance' and 'total_amount' was removed since it no longer seemed necessary after the feature engineering and will give noise. So after making these changes especially dealing with outliers, the Linear Regression was able to perform better since it sensitive extreme values. Thus, the linear model was able to capture the relationship between tip_amount and predictors. (This also decreased my runtime greatly so I stuck with it)

Classification Model predict high_tip: Initially the Random Forest Classifier performed the best compared to Logistic Regression for the same reasons state prior. After the additional cleaning Logistic performed better. However, overall the Neural Network performed the best. This is supported by it having the highest Accuracy (0.7711), Precision (0.7689), Recall(0.9989), F1 (0.8689) and AUC-ROC (0.6139). It performed slightly better than Logistic Regression across all metrics while Random Forest performed the worst. The neural network was the best since it is capable of capturing more complex relationships between the features. However, since the improvement was small this suggests that the classification problem is still a bit difficult with the current features. This may be because tipping behavior is highly variable and influenced by factors not captured in the dataset, making it difficult to clearly separate high-tip and non-high-tip trips.


(b) What features are most predictive of tip amount and whether this aligns with your intuition
Regression Model
For regression, similar fare and trip-related variables such as speed are likely among the most predictive of actual tip amount. Generally, fare, trip and location related features are intuitively reasonable predictors because tipping depends on trip cost, trip experience, and context.

Classification Model
The top 10 features that were most predictive of tip amount were num__log_trip_distance (-0.403691), num__Airport_fee (0.284287), num__congestion_surcharge (0.187861), num__pickup_borough_Bronx (-0.133322)    num__trip_duration_minutes (-0.125817), num__pickup_borough_Manhattan, (0.096826), num__tolls_amount (0.092803), num__fare_amount (0.087152), num__pickup_borough_Brooklyn (-0.082462) and num__extra (0.069179). The strongest positive being num_Airport_fee and strongest negative being num__log_trip_distance. The negative effect of num_log_trip_distance and num__trip_duration_minutes shows that persons don't tip higher according to how long their trips are and will more than likely tip higher when the trips are shorter since they were able to reach their destination quickly.
It makes sense that trips with airport fees, congestion surcharges, tolls, and higher fares are often associated with more expensive or structured trips, which may lead to higher tips. The variety in negative and positive values for Boroughs also makes sense since the tip behaviour will vary by trip location.

(c) Limitations of your models (e.g., data leakage concerns, feature limitations, dataset biases)
Limitations:
- One major data leakage concern was with total_amount column because it would artificially inflate performance because they indirectly contain tip information. Hence, why I saw necessary to remove it to get more reliable and realistic results.
- Another limitation would be the lack of certain features in the dataset. Tip behaviour is influenced by additional features that was not present such as driver's behaviour during the trip, service quality, purpose of the trip, passenger income etc. These features would have increase the predictive power of the model tremendously.
- Dataset was bias with respect to the payment type being only card.

(d) Potential improvements you would make given more time or data
- Given more time I would work more on the preprocessing so that all numeric features are included such as pickup_hour, pickup_weekday and pickup_day_of_week was excluded because the data type is int8 and only data type int64 and float64. Was not able to add due to time and computational constraints. (I not risking it with this runtime right now..)
- I would work more on hypertuning especially with Neural Network and try more than 2 hidden layers
- I would have done the optional task: Use SHAP values to explain individual predictions for 3 sample trips

(e) A brief discussion comparing the neural network approach to the traditional ML models for this particular problem
For this problem, the Neural network performed slightly better than the traditional classification models by a small margin. It achieved the best Accuracy, F1, and AUC-ROC among the classifiers, showing that it was able to capture some additional nonlinear structure in the data. Early stopping also helped ensure stable training without overfitting. Logistic Regression performed nearly as well while being much simpler, faster to train, and easier to interpret through coefficients. This suggests that that the available features used limit how much a more flexible model can improve performance.


AI USAGE
- Used chat gpt to look into log transformed distance and what method to use.
  - Learned that this is done since usually the distance is skewed to the right so log transform comes in to reduce the effect of the extremely large values.
  - Chose log1p() or log() since it handles 0 values safely.
- Used chatgpt to understand when to use one hot encoding and when to use label encoding.
  - While label encoding would have less columns compared to one hot, I decided to go with one hot encoding since it will work across all the models for this assignment. (specifically logistic and linear regression model and neural network. These models would have the risk of misreading label encoded categories.)
  - How to do one hot encoding
- To confirm how to split the data in a 3 ratio
- Ask for a comparison between StandardScaler and MinMaxScaler to help me decide which one to use, went with StandardScaler since it is less sensitive to extreme values and is better for Linear and Logistic Regression
- Asked about RandomizedSearch hyperparameters to get a better understanding and also asked what hyperparamters can be used for Logistic Regression model
- Since BCEWithLogitsLoss was used I got error when evaluating with loader so I used chatgpt to see why turns out I needed to apply sigmoid to get probabilities otherwise I will have logits
- Was back and forth between labs and chatgpt for the plots for te sake of time
- README.md